# PETA – Photo Album Event Recognition


## Cell 1 - Clone source từ GitHub & kiểm tra môi trường

In [1]:
import subprocess, sys, os, shutil

GITHUB_URL = 'https://github.com/gbao23127157/Final_ComputerVision_PETA.git'
CLONE_DIR  = '/kaggle/working/MyCode'

os.chdir('/kaggle/working')

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

subprocess.run(['git', 'clone', GITHUB_URL, CLONE_DIR], check=True)

sys.path.insert(0, CLONE_DIR)
os.chdir(CLONE_DIR)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

Cloning into '/kaggle/working/MyCode'...
Updating files: 100% (841/841), done.


PyTorch : 2.10.0+cu128
GPU     : Tesla T4


In [2]:
%cd /kaggle/working/MyCode/Source

/kaggle/working/MyCode/Source


## Cell 2 - Chuẩn bị cấu trúc thư mục album

In [3]:
IMAGES_DIR  = '/kaggle/input/datasets/huynhtien123/pec-privare/pec/images'
RAW_ALBUMS  = '/kaggle/working/data/raw_albums'
FEATURE_DIR = '/kaggle/working/data/features'
os.makedirs(RAW_ALBUMS,  exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

# Tạo symlink flat 
for cls_name in os.listdir(IMAGES_DIR):
    cls_dir = os.path.join(IMAGES_DIR, cls_name)
    if not os.path.isdir(cls_dir): continue
    for album_id in os.listdir(cls_dir):
        src = os.path.join(cls_dir, album_id)
        dst = os.path.join(RAW_ALBUMS, f'{cls_name}_{album_id}')
        pt  = os.path.join(FEATURE_DIR, f'{cls_name}_{album_id}.pt')
        if os.path.isdir(src) and not os.path.exists(dst) and not os.path.exists(pt):
            os.symlink(src, dst)

remaining = [
    d for d in os.listdir(RAW_ALBUMS)
    if not os.path.exists(os.path.join(FEATURE_DIR, f'{d}.pt'))
]
print(f'Album cần trích xuất : {len(remaining)}')
print(f'Album đã có .pt      : {len(os.listdir(FEATURE_DIR))}')

Album cần trích xuất : 0
Album đã có .pt      : 804


## Cell 3 - Trích xuất đặc trưng ResNet50

In [4]:
if remaining:
    from extract_features import extract_and_save_features
    extract_and_save_features(RAW_ALBUMS, FEATURE_DIR)
else:
    print('Tất cả features đã có, bỏ qua.')

Tất cả features đã có, bỏ qua.


## Cell 4 - Train PETA

In [5]:
os.makedirs('/kaggle/working/Release', exist_ok=True)
os.makedirs('/kaggle/working/Docs',    exist_ok=True)

# Patch 3 đường dẫn trong train.py về /kaggle/working/
with open('train.py') as f:
    src = f.read()

src = src \
    .replace('LOG_PATH = "../Docs/training_log.txt"',
             'LOG_PATH = "/kaggle/working/Docs/training_log.txt"') \
    .replace('SAVE_PATH = "../Release/best_peta_model.pth"',
             'SAVE_PATH = "/kaggle/working/Release/best_peta_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"') \
    .replace('verbose=True', '')

with open('/tmp/train_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/train_kaggle.py

2026-04-07 17:09:44 - INFO - Khởi động huấn luyện PETA 
2026-04-07 17:09:44 - INFO - Dữ liệu ép chuẩn: Train: 643 albums | Test: 161 albums
2026-04-07 17:09:46 - INFO - --- Epoch 1/30 ---


2026-04-07 17:09:51 - INFO - Train - Loss: 2.5621 | Accuracy: 0.1757 | mAP: 0.1606


2026-04-07 17:09:52 - INFO - Test - Loss: 1.7951 | Accuracy: 0.4969 | mAP: 0.7039
2026-04-07 17:09:52 - INFO - Learning Rate hiện tại: 0.010000


2026-04-07 17:09:52 - INFO - -> Đã lưu mô hình tốt nhất với Accuracy: 0.4969 tại /kaggle/working/Release/best_peta_model.pth
2026-04-07 17:09:52 - INFO - --- Epoch 2/30 ---


2026-04-07 17:09:57 - INFO - Train - Loss: 2.0587 | Accuracy: 0.3655 | mAP: 0.3834


...

2026-04-07 17:12:15 - INFO - Test - Loss: 1.3270 | Accuracy: 0.8571 | mAP: 0.9275
2026-04-07 17:12:15 - INFO - Learning Rate hiện tại: 0.000100
2026-04-07 17:12:15 - INFO - --- Epoch 29/30 ---


2026-04-07 17:12:19 - INFO - Train - Loss: 0.7196 | Accuracy: 0.9751 | mAP: 0.9893


2026-04-07 17:12:20 - INFO - Test - Loss: 1.2828 | Accuracy: 0.8509 | mAP: 0.9253
2026-04-07 17:12:20 - INFO - Learning Rate hiện tại: 0.000100
2026-04-07 17:12:20 - INFO - --- Epoch 30/30 ---


2026-04-07 17:12:24 - INFO - Train - Loss: 0.7149 | Accuracy: 0.9642 | mAP: 0.9953


2026-04-07 17:12:25 - INFO - Test - Loss: 1.2798 | Accuracy: 0.8571 | mAP: 0.9393
2026-04-07 17:12:25 - INFO - Learning Rate hiện tại: 0.000100
2026-04-07 17:12:25 - INFO - Hoàn tất quá trình huấn luyện!


## Cell 5 - Train baseline

In [6]:
os.makedirs('/kaggle/working/Release', exist_ok=True)
os.makedirs('/kaggle/working/Docs',    exist_ok=True)

# Patch đường dẫn trong train_baseline.py về /kaggle/working/
with open('train_baseline.py') as f:
    src = f.read()

src = src \
    .replace('LOG_PATH = "../Docs/training_baseline_log.txt"',
             'LOG_PATH = "/kaggle/working/Docs/training_baseline_log.txt"') \
    .replace('LOG_PATH = "../Docs/training_log.txt"',
             'LOG_PATH = "/kaggle/working/Docs/training_baseline_log.txt"') \
    .replace('SAVE_PATH = "../Release/best_baseline_model.pth"',
             'SAVE_PATH = "/kaggle/working/Release/best_baseline_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"') \
    .replace('verbose=True', '')

with open('/tmp/train_baseline_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/train_baseline_kaggle.py

2026-04-07 17:12:31 - INFO - --- Epoch 1/20 ---


2026-04-07 17:12:31 - INFO - Train - Loss: 1.9091 | Accuracy: 0.4630 | mAP: 0.5142


2026-04-07 17:12:31 - INFO - Val - Loss: 1.8073 | Accuracy: 0.7209 | mAP: 0.7785
2026-04-07 17:12:31 - INFO - -> Đã lưu mô hình tốt nhất với Accuracy: 0.7209 tại /kaggle/working/Release/best_baseline_model.pth
2026-04-07 17:12:31 - INFO - --- Epoch 2/20 ---


2026-04-07 17:12:32 - INFO - Train - Loss: 1.0449 | Accuracy: 0.7860 | mAP: 0.8417


2026-04-07 17:12:32 - INFO - Val - Loss: 0.9705 | Accuracy: 0.8062 | mAP: 0.8578
2026-04-07 17:12:32 - INFO - -> Đã lưu mô hình tốt nhất với Accuracy: 0.8062 tại /kaggle/working/Release/best_baseline_model.pth
2026-04-07 17:12:32 - INFO - --- Epoch 3/20 ---


...

2026-04-07 17:12:42 - INFO - Val - Loss: 0.4841 | Accuracy: 0.8682 | mAP: 0.9199
2026-04-07 17:12:42 - INFO - --- Epoch 19/20 ---


2026-04-07 17:12:42 - INFO - Train - Loss: 0.1195 | Accuracy: 0.9922 | mAP: 0.9998


2026-04-07 17:12:42 - INFO - Val - Loss: 0.4466 | Accuracy: 0.8837 | mAP: 0.9235
2026-04-07 17:12:42 - INFO - --- Epoch 20/20 ---


2026-04-07 17:12:43 - INFO - Train - Loss: 0.1134 | Accuracy: 0.9883 | mAP: 0.9986


2026-04-07 17:12:43 - INFO - Val - Loss: 0.5092 | Accuracy: 0.8450 | mAP: 0.9273


## Cell 6 - Evaluate trên Test set (PETA)

In [7]:
with open('evaluate.py') as f:
    src = f.read()

src = src \
    .replace('MODEL_WEIGHTS = "../Release/best_peta_model.pth"',
             'MODEL_WEIGHTS = "/kaggle/working/Release/best_peta_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"')

with open('/tmp/evaluate_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/evaluate_kaggle.py

Đang chuẩn bị dữ liệu Test...
-> Đã tải thành công file trọng số best_peta_model.pth!

BẮT ĐẦU ĐÁNH GIÁ 5 LẦN 



   Hoàn thành Lần 1 | Accuracy: 87.58% | mAP: 91.79%


   Hoàn thành Lần 2 | Accuracy: 86.34% | mAP: 92.98%


   Hoàn thành Lần 3 | Accuracy: 85.09% | mAP: 92.60%


   Hoàn thành Lần 4 | Accuracy: 84.47% | mAP: 91.66%


   Hoàn thành Lần 5 | Accuracy: 85.09% | mAP: 92.31%

 BÁO CÁO KẾT QUẢ PETA (TRUNG BÌNH SAU 5 LẦN)
  Accuracy : 85.71% ± 1.11%
  mAP      : 92.27% ± 0.49%


## Cell 7 - Evaluate trên Test set (Baseline)

In [8]:
with open('evaluate_baseline.py') as f:
    src = f.read()

# Patch đường dẫn trọng số của mô hình baseline
src = src \
    .replace('MODEL_WEIGHTS = "../Release/best_baseline_model.pth"',
             'MODEL_WEIGHTS = "/kaggle/working/Release/best_baseline_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"')

with open('/tmp/evaluate_baseline_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/evaluate_baseline_kaggle.py

Đang chuẩn bị dữ liệu Test...
-> Đã tải thành công file trọng số /kaggle/working/Release/best_baseline_model.pth!

BẮT ĐẦU ĐÁNH GIÁ 5 LẦN...



   Hoàn thành Lần 1 | Accuracy: 85.09% | mAP: 90.75%


   Hoàn thành Lần 2 | Accuracy: 84.47% | mAP: 90.33%


   Hoàn thành Lần 3 | Accuracy: 86.34% | mAP: 90.47%


   Hoàn thành Lần 4 | Accuracy: 86.96% | mAP: 91.36%


   Hoàn thành Lần 5 | Accuracy: 83.85% | mAP: 91.18%

 BÁO CÁO KẾT QUẢ BASELINE (TRUNG BÌNH SAU 5 LẦN)
  Accuracy : 85.34% ± 1.15%
  mAP      : 90.82% ± 0.40%
